# Chapter 18 Computational Lab
## Actuarial Mathematics: Risk, Premiums and Reinsurance

This notebook accompanies Chapter 18 of *Probability Theory with Python and AI*.

The chapter translates probability into actuarial decisions:

$$
\boxed{
\text{loss model}
\longrightarrow
\text{premium}
\longrightarrow
\text{aggregate risk}
\longrightarrow
\text{risk transfer}
\longrightarrow
\text{tail-risk decision}.
}
$$

### Learning goals

By the end of the lab you should be able to:

1. model an individual insurance loss by a nonnegative random variable;
2. use the survival function and tail-integral identity;
3. compare net, expected-value, variance, standard-deviation and exponential premium principles;
4. derive the small-risk expansion of the exponential premium;
5. compute the mean, variance, MGF and characteristic function of random sums;
6. specialize these formulas to compound Poisson aggregate losses;
7. use the compound Poisson loss process and its compensated martingale;
8. compute expected payments under deductibles and policy limits;
9. use the stop-loss transform and its monotonicity/convexity;
10. distinguish quota-share, per-loss excess-of-loss and aggregate stop-loss reinsurance;
11. compute VaR as a quantile and explain why VaR need not be subadditive;
12. use the optimization definition of CVaR / Expected Shortfall;
13. understand quantile minimizers and the average-of-quantiles formula;
14. use coherence properties of CVaR;
15. solve the exponential excess-of-loss retention problem under a CVaR-plus-premium objective;
16. interpret the Cramér--Lundberg surplus process and ruin time;
17. distinguish positive long-run drift from zero ruin probability;
18. audit AI-generated actuarial claims.

> **Decision principle.** No premium principle, risk measure or reinsurance optimizer is universally “correct.” Each is relative to the stated model, objective and assumptions.


## 0. Setup


In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def exponential_var(alpha, beta):
    return -math.log(1-alpha)/beta


def exponential_cvar(alpha, beta):
    return exponential_var(alpha,beta) + 1/beta


def stop_loss_exponential(d, beta):
    return math.exp(-beta*d)/beta


def limited_expected_exponential(u, beta):
    return (1-math.exp(-beta*u))/beta


def capped_excess_expected_exponential(d, u, beta):
    return (
        math.exp(-beta*d)
        -
        math.exp(-beta*(d+u))
    )/beta


def compound_poisson_moments(lam, mean_x, second_x):
    return lam*mean_x, lam*second_x


def simulate_compound_poisson(lam, mean_severity, n_years, seed=2026):
    rng = np.random.default_rng(seed)
    counts = rng.poisson(lam,size=n_years)
    losses = np.zeros(n_years)

    for m in np.unique(counts):
        idx = np.where(counts == m)[0]
        if m > 0:
            claims = rng.exponential(
                mean_severity,
                size=(len(idx),m),
            )
            losses[idx] = claims.sum(axis=1)

    return counts, losses


def empirical_var(x, alpha):
    return float(np.quantile(
        np.asarray(x,dtype=float),
        alpha,
        method="higher",
    ))


def empirical_cvar(x, alpha):
    x = np.asarray(x,dtype=float)
    q = empirical_var(x,alpha)
    return q + np.mean(np.maximum(x-q,0))/(1-alpha)


def exp_retention_regime(beta, alpha, theta):
    threshold = (1+theta)*(1-alpha)

    if threshold < 1:
        return "finite", math.log(1+theta)/beta
    if abs(threshold-1) < 1e-12:
        return "flat-tail", exponential_var(alpha,beta)
    return "no-reinsurance", math.inf


def retained_cvar_exponential(d, alpha, beta):
    q = exponential_var(alpha,beta)

    if d <= q:
        return d

    return (
        q + 1/beta
        - math.exp(-beta*d)/((1-alpha)*beta)
    )


def retention_objective(d, alpha, beta, theta):
    retained = retained_cvar_exponential(d,alpha,beta)
    reinsurance_premium = (1+theta)*stop_loss_exponential(d,beta)
    return retained + reinsurance_premium


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))
    for line in latex_lines:
        display(Math(line))
    if note:
        display(Markdown(note))


## 1. Individual loss model

An individual loss is a nonnegative random variable

$$
\boxed{
X\ge0.
}
$$

Its cdf is

$$
F_X(x)=P(X\le x),
$$

and its survival function is

$$
\boxed{
\overline F_X(x)=P(X>x)=1-F_X(x).
}
$$

Moment and premium assumptions are stated separately, so heavy-tailed losses are not excluded at the definition stage.


### Tail-integral formula

If $X\ge0$ and $E[X]<\infty$,

$$
\boxed{
E[X]
=
\int_0^\infty
\overline F_X(x)\,dx.
}
$$

The area under the survival curve is the net premium.


### Exponential severity

For

$$
X\sim\operatorname{Exp}(\beta),
$$

$$
\overline F_X(x)=e^{-\beta x},
$$

and

$$
\boxed{
E[X]=1/\beta.
}
$$


In [ ]:
sev_mean = widgets.FloatSlider(
    value=2000,
    min=200,
    max=10000,
    step=200,
    description="mean",
)
sev_output = widgets.Output()

def update_severity(*_):
    with sev_output:
        clear_output(wait=True)
        mean = sev_mean.value
        beta = 1/mean
        x = np.linspace(0,5*mean,700)
        surv = np.exp(-beta*x)

        fig, ax = plt.subplots(figsize=(8,3.4))
        ax.plot(x,surv)
        ax.fill_between(x,0,surv,alpha=0.2)
        ax.set_xlabel("loss x")
        ax.set_ylabel("survival probability")
        ax.set_title("Exponential survival curve")
        plt.show()

        display(Math(r"\int_0^\infty \overline F_X(x)\,dx=" + f"{mean:.2f}"))

sev_mean.observe(update_severity,names="value")
display(widgets.VBox([sev_mean,sev_output]))
update_severity()


## 2. Premium principles

A premium principle assigns a monetary value $\Pi(X)$ to a loss.

The main rules in the chapter are:

$$
\boxed{
\Pi_0(X)=E[X]
}
$$

for the net premium,

$$
\boxed{
\Pi_{\mathrm{EV}}(X)
=
(1+\theta)E[X]
}
$$

for the expected-value principle,

$$
\Pi_{\mathrm{Var}}(X)
=
E[X]+a\operatorname{Var}(X),
$$

$$
\Pi_{\mathrm{SD}}(X)
=
E[X]+b\sqrt{\operatorname{Var}(X)},
$$

and, when the MGF is finite at $\gamma>0$,

$$
\boxed{
\Pi_{\mathrm{exp}}(X)
=
\frac1\gamma
\log E[e^{\gamma X}].
}
$$


### Structural properties of the exponential premium

Whenever the required exponential moments are finite,

$$
\boxed{
\Pi_{\mathrm{exp}}(X+c)
=
\Pi_{\mathrm{exp}}(X)+c,
}
$$

and for independent $X,Y$,

$$
\boxed{
\Pi_{\mathrm{exp}}(X+Y)
=
\Pi_{\mathrm{exp}}(X)
+
\Pi_{\mathrm{exp}}(Y).
}
$$


### Small-risk expansion

If the MGF is finite in a neighborhood of zero,

$$
\boxed{
\Pi_{\mathrm{exp}}(X)
=
E[X]
+
\frac{\gamma}{2}\operatorname{Var}(X)
+
o(\gamma)
\qquad
(\gamma\downarrow0).
}
$$

Thus the net premium is the leading term and variance supplies the first risk correction.


In [ ]:
premium_beta = widgets.FloatSlider(value=0.001,min=0.0002,max=0.003,step=0.0001,description="beta")
premium_gamma = widgets.FloatSlider(value=0.0002,min=0.00002,max=0.0009,step=0.00002,description="gamma")
premium_output = widgets.Output()

def update_exp_premium(*_):
    with premium_output:
        clear_output(wait=True)
        beta = premium_beta.value
        gamma = premium_gamma.value

        if gamma >= beta:
            display(Markdown("Choose $\\gamma<\\beta$ so the exponential MGF is finite."))
            return

        exact = math.log(beta/(beta-gamma))/gamma
        approx = 1/beta + gamma/(2*beta*beta)

        display(Math(r"\Pi_{\mathrm{exp}}(X)=" + f"{exact:.4f}"))
        display(Math(r"\text{small-}\gamma\text{ approximation}=" + f"{approx:.4f}"))
        display(Math(r"\text{absolute difference}=" + f"{abs(exact-approx):.4f}"))

for c in (premium_beta,premium_gamma):
    c.observe(update_exp_premium,names="value")

display(widgets.VBox([
    widgets.HBox([premium_beta,premium_gamma]),
    premium_output,
]))
update_exp_premium()


## 3. Frequency, severity and aggregate loss

Let $N$ be the claim count and let $X_1,X_2,\ldots$ be iid nonnegative severities independent of $N$.

Define

$$
\boxed{
S
=
\sum_{i=1}^{N}X_i,
}
$$

with $S=0$ when $N=0$.


### Mean and variance of a random sum

If $E[N^2]<\infty$ and

$$
\mu=E[X_1],
\qquad
\sigma^2=\operatorname{Var}(X_1)<\infty,
$$

then

$$
\boxed{
E[S]=E[N]\mu,
}
$$

and

$$
\boxed{
\operatorname{Var}(S)
=
E[N]\sigma^2
+
\operatorname{Var}(N)\mu^2.
}
$$

The two terms correspond to severity variation and count variation.


In [ ]:
freq_mean = widgets.FloatSlider(value=12,min=1,max=40,step=1,description="E[N]")
freq_var = widgets.FloatSlider(value=12,min=1,max=60,step=1,description="Var(N)")
sev_mu = widgets.FloatSlider(value=2000,min=100,max=10000,step=100,description="mu")
sev_sd = widgets.FloatSlider(value=2000,min=100,max=10000,step=100,description="sigma")
random_sum_output = widgets.Output()

def update_random_sum(*_):
    with random_sum_output:
        clear_output(wait=True)
        EN = freq_mean.value
        VN = freq_var.value
        mu = sev_mu.value
        sigma = sev_sd.value

        mean_s = EN*mu
        var_s = EN*sigma**2 + VN*mu**2

        display(Math(r"E[S]=" + f"{mean_s:,.2f}"))
        display(Math(r"\operatorname{Var}(S)=" + f"{var_s:,.2f}"))

for c in (freq_mean,freq_var,sev_mu,sev_sd):
    c.observe(update_random_sum,names="value")

display(widgets.VBox([
    widgets.HBox([freq_mean,freq_var]),
    widgets.HBox([sev_mu,sev_sd]),
    random_sum_output,
]))
update_random_sum()


### Random-sum transforms

Let

$$
\widetilde G_N(z)
=
\sum_{n=0}^{\infty}P(N=n)z^n,
\qquad
z\ge0.
$$

Whenever $M_X(t)$ is finite,

$$
\boxed{
E[e^{tS}]
=
\widetilde G_N(M_X(t)).
}
$$

The characteristic-function form requires no positive exponential moment:

$$
\boxed{
\varphi_S(t)
=
G_N(\varphi_X(t)).
}
$$


## 4. Compound Poisson aggregate loss

If

$$
N\sim\operatorname{Poisson}(\lambda),
$$

then

$$
S=\sum_{i=1}^NX_i
$$

is a compound Poisson aggregate loss.

If $E[X_1^2]<\infty$,

$$
\boxed{
E[S]
=
\lambda E[X_1],
}
$$

and

$$
\boxed{
\operatorname{Var}(S)
=
\lambda E[X_1^2].
}
$$


The transform formulas become

$$
\boxed{
M_S(t)
=
\exp\{
\lambda(M_X(t)-1)
\},
}
$$

and

$$
\boxed{
\varphi_S(t)
=
\exp\{
\lambda(\varphi_X(t)-1)
\}.
}
$$


### Poisson frequency with exponential severity

If

$$
N\sim\operatorname{Poisson}(10),
\qquad
X_i\sim\operatorname{Exp}(1/1000),
$$

then

$$
\boxed{
E[S]=10{,}000,
}
$$

and

$$
\boxed{
\operatorname{Var}(S)=20{,}000{,}000.
}
$$


In [ ]:
cp_N = widgets.IntSlider(value=50000,min=5000,max=150000,step=5000,description="years")
cp_output = widgets.Output()

def update_compound_sim(*_):
    with cp_output:
        clear_output(wait=True)
        n_years = cp_N.value
        _, S = simulate_compound_poisson(
            lam=10,
            mean_severity=1000,
            n_years=n_years,
            seed=2026,
        )

        display(Math(r"\widehat E[S]=" + f"{S.mean():,.2f}"))
        display(Math(r"E[S]=10{,}000"))
        display(Math(r"\widehat{\operatorname{Var}}(S)=" + f"{S.var():,.2f}"))
        display(Math(r"\operatorname{Var}(S)=20{,}000{,}000"))

cp_N.observe(update_compound_sim,names="value")
display(widgets.VBox([cp_N,cp_output]))
update_compound_sim()


## 5. Compound Poisson loss process

Let $N(t)$ be a Poisson process of rate $\lambda$ and define

$$
\boxed{
S(t)
=
\sum_{i=1}^{N(t)}X_i.
}
$$

With

$$
\mathcal F_t^{N,S}
=
\sigma(
N(u),S(u):0\le u\le t
),
$$

the chapter records

$$
E[S(t)]
=
\lambda t\mu,
$$

$$
\operatorname{Var}(S(t))
=
\lambda tE[X_1^2],
$$

and

$$
\boxed{
\varphi_{S(t)}(u)
=
\exp\{
\lambda t(\varphi_X(u)-1)
\}.
}
$$


### Compensated aggregate-loss martingale

If $\mu=E[X_1]<\infty$,

$$
\boxed{
M(t)
=
S(t)-\lambda t\mu
}
$$

is a martingale with respect to the observed count-and-loss filtration.


If the annual arrival rate is $12$ and mean severity is $2000$, the expected aggregate loss over one quarter is

$$
\boxed{
12\cdot\frac14\cdot2000
=
6000.
}
$$


## 6. Deductibles and policy limits

For a deductible $d$,

$$
\boxed{
Y_d=(X-d)^+.
}
$$

For a policy limit $u$,

$$
\boxed{
Y_u=X\wedge u.
}
$$

With both deductible and maximum insurer payment,

$$
\boxed{
Y_{d,u}
=
\min\{
(X-d)^+,u
\}.
}
$$


### Expected payments from the tail

For nonnegative integrable $X$,

$$
\boxed{
E[(X-d)^+]
=
\int_d^\infty
\overline F_X(x)\,dx,
}
$$

$$
\boxed{
E[X\wedge u]
=
\int_0^u
\overline F_X(x)\,dx,
}
$$

and

$$
\boxed{
E[
\min\{(X-d)^+,u\}
]
=
\int_d^{d+u}
\overline F_X(x)\,dx.
}
$$


### Exponential loss

For $X\sim\operatorname{Exp}(\beta)$,

$$
\boxed{
E[(X-d)^+]
=
\frac{e^{-\beta d}}{\beta},
}
$$

and

$$
\boxed{
E[X\wedge d]
=
\frac{1-e^{-\beta d}}{\beta}.
}
$$

Their sum is $E[X]=1/\beta$.


In [ ]:
ded_mean = widgets.FloatSlider(value=2000,min=200,max=10000,step=200,description="mean")
ded_d = widgets.FloatSlider(value=1000,min=0,max=10000,step=200,description="d")
ded_output = widgets.Output()

def update_deductible(*_):
    with ded_output:
        clear_output(wait=True)
        mean = ded_mean.value
        beta = 1/mean
        d = ded_d.value

        excess = stop_loss_exponential(d,beta)
        capped = limited_expected_exponential(d,beta)

        display(Math(r"E[(X-d)^+]=" + f"{excess:.4f}"))
        display(Math(r"E[X\wedge d]=" + f"{capped:.4f}"))
        display(Math(r"\text{sum}=" + f"{excess+capped:.4f}"))

for c in (ded_mean,ded_d):
    c.observe(update_deductible,names="value")

display(widgets.VBox([
    widgets.HBox([ded_mean,ded_d]),
    ded_output,
]))
update_deductible()


## 7. Stop-loss transform

The stop-loss transform is

$$
\boxed{
\pi_X(d)
=
E[(X-d)^+].
}
$$

It is decreasing and convex in $d$.

Its one-sided slopes recover the survival probabilities; when the cdf is continuous,

$$
\boxed{
\pi_X'(d)
=
-\overline F_X(d).
}
$$


In [ ]:
sl_mean = widgets.FloatSlider(value=2000,min=200,max=8000,step=200,description="mean")
sl_output = widgets.Output()

def update_stoploss_curve(*_):
    with sl_output:
        clear_output(wait=True)
        mean = sl_mean.value
        beta = 1/mean
        d = np.linspace(0,5*mean,600)
        pi = np.exp(-beta*d)/beta

        fig, ax = plt.subplots(figsize=(8,3.4))
        ax.plot(d,pi)
        ax.set_xlabel("retention d")
        ax.set_ylabel("stop-loss transform")
        ax.set_title("Decreasing convex stop-loss transform")
        plt.show()

sl_mean.observe(update_stoploss_curve,names="value")
display(widgets.VBox([sl_mean,sl_output]))
update_stoploss_curve()


## 8. Reinsurance as risk transfer

A reinsurance rule decomposes the loss:

$$
\boxed{
X=R(X)+C(X),
}
$$

where $R(X)$ is retained and $C(X)$ is ceded.


### Quota share

For $q\in[0,1]$,

$$
\boxed{
C_q(X)=qX,
\qquad
R_q(X)=(1-q)X.
}
$$

If the moments exist,

$$
E[R_q(X)]
=
(1-q)E[X],
$$

and

$$
\boxed{
\operatorname{Var}(R_q(X))
=
(1-q)^2
\operatorname{Var}(X).
}
$$


### Excess of loss

For retention $d$,

$$
\boxed{
R_d(X)=X\wedge d,
\qquad
C_d(X)=(X-d)^+.
}
$$

The same positive-part transformation can be applied to annual aggregate loss $S$, but that is an **aggregate stop-loss** contract rather than a per-loss excess-of-loss treaty.


In [ ]:
q_slider = widgets.FloatSlider(value=0.35,min=0,max=1,step=0.05,description="q ceded")
q_output = widgets.Output()

def update_quota_share(*_):
    with q_output:
        clear_output(wait=True)
        q = q_slider.value
        mean = 10000
        var = 25_000_000
        retained_mean = (1-q)*mean
        retained_var = (1-q)**2*var

        display(Math(r"E[R]=" + f"{retained_mean:,.2f}"))
        display(Math(r"\operatorname{Var}(R)=" + f"{retained_var:,.2f}"))

q_slider.observe(update_quota_share,names="value")
display(widgets.VBox([q_slider,q_output]))
update_quota_share()


## 9. Value at Risk

For confidence level $\alpha\in(0,1)$,

$$
\boxed{
\operatorname{VaR}_\alpha(X)
=
q_X(\alpha)
}
$$

is the generalized-inverse quantile.

VaR locates the upper-tail cutoff. It does **not** measure how severe the losses beyond that cutoff are.


For $X\sim\operatorname{Exp}(\beta)$,

$$
\boxed{
\operatorname{VaR}_\alpha(X)
=
-\frac{\log(1-\alpha)}{\beta}.
}
$$


In [ ]:
var_alpha = widgets.FloatSlider(value=0.99,min=0.8,max=0.999,step=0.001,description="alpha")
var_mean = widgets.FloatSlider(value=1000,min=100,max=5000,step=100,description="mean")
var_output = widgets.Output()

def update_var(*_):
    with var_output:
        clear_output(wait=True)
        alpha = var_alpha.value
        beta = 1/var_mean.value
        q = exponential_var(alpha,beta)
        display(Math(r"\operatorname{VaR}_\alpha=" + f"{q:.4f}"))

for c in (var_alpha,var_mean):
    c.observe(update_var,names="value")

display(widgets.VBox([
    widgets.HBox([var_alpha,var_mean]),
    var_output,
]))
update_var()


### VaR need not be subadditive

Let independent $X,Y$ satisfy

$$
P(X=1)=P(Y=1)=0.04,
$$

and

$$
P(X=0)=P(Y=0)=0.96.
$$

At level $0.95$,

$$
\operatorname{VaR}_{0.95}(X)
=
\operatorname{VaR}_{0.95}(Y)
=
0,
$$

but

$$
\boxed{
\operatorname{VaR}_{0.95}(X+Y)=1.
}
$$

Thus VaR can fail the diversification inequality.


In [ ]:
p0 = 0.96
display(Math(r"P(X+Y=0)=" + f"{p0**2:.4f}"))
display(Math(r"P(X+Y\le1)=" + f"{1-0.04**2:.4f}"))


## 10. CVaR / Expected Shortfall

For integrable $X$ and $\alpha\in(0,1)$,

$$
\boxed{
\operatorname{CVaR}_\alpha(X)
=
\inf_{z\in\mathbb R}
\left\{
z+
\frac{1}{1-\alpha}
E[(X-z)^+]
\right\}.
}
$$

This definition remains unambiguous even when the loss distribution has atoms.


### Quantile characterization of minimizers

For

$$
h(z)
=
z+
\frac{1}{1-\alpha}
E[(X-z)^+],
$$

the one-sided derivatives are

$$
\boxed{
h'_-(z)
=
\frac{F_X(z-)-\alpha}{1-\alpha},
}
$$

and

$$
\boxed{
h'_+(z)
=
\frac{F_X(z)-\alpha}{1-\alpha}.
}
$$

Hence

$$
\boxed{
z\in\operatorname*{argmin}h
\iff
F_X(z-)\le\alpha\le F_X(z).
}
$$


### Continuous-cdf formula

If the cdf is continuous and

$$
q_\alpha=\operatorname{VaR}_\alpha(X),
$$

then

$$
\boxed{
\operatorname{CVaR}_\alpha(X)
=
q_\alpha
+
\frac1{1-\alpha}
E[(X-q_\alpha)^+].
}
$$

Equivalently,

$$
\boxed{
\operatorname{CVaR}_\alpha(X)
=
q_\alpha
+
\frac1{1-\alpha}
\int_{q_\alpha}^{\infty}
\overline F_X(x)\,dx.
}
$$


For a continuous cdf,

$$
\boxed{
\operatorname{CVaR}_\alpha(X)
=
E[
X\mid X>q_\alpha
].
}
$$

This conditional-tail formula should not be extended blindly to distributions with atoms.


### Exponential CVaR

For $X\sim\operatorname{Exp}(\beta)$,

$$
\boxed{
\operatorname{CVaR}_\alpha(X)
=
-\frac{\log(1-\alpha)}{\beta}
+
\frac1\beta.
}
$$

The exponential memoryless property makes the tail excess equal to one additional mean severity.


In [ ]:
cvar_alpha = widgets.FloatSlider(value=0.99,min=0.8,max=0.999,step=0.001,description="alpha")
cvar_mean = widgets.FloatSlider(value=1000,min=100,max=5000,step=100,description="mean")
cvar_output = widgets.Output()

def update_cvar(*_):
    with cvar_output:
        clear_output(wait=True)
        alpha = cvar_alpha.value
        beta = 1/cvar_mean.value
        q = exponential_var(alpha,beta)
        c = exponential_cvar(alpha,beta)

        display(Math(r"\operatorname{VaR}_\alpha=" + f"{q:.4f}"))
        display(Math(r"\operatorname{CVaR}_\alpha=" + f"{c:.4f}"))
        display(Math(r"\operatorname{CVaR}-\operatorname{VaR}=" + f"{c-q:.4f}"))

for control in (cvar_alpha,cvar_mean):
    control.observe(update_cvar,names="value")

display(widgets.VBox([
    widgets.HBox([cvar_alpha,cvar_mean]),
    cvar_output,
]))
update_cvar()


### CVaR as average quantile

For any integrable loss,

$$
\boxed{
\operatorname{CVaR}_\alpha(X)
=
\frac1{1-\alpha}
\int_\alpha^1
q_X(u)\,du.
}
$$

This representation remains valid in the presence of atoms.


### Coherence

CVaR is a coherent risk measure: it is

- monotone;
- translation invariant;
- positively homogeneous;
- subadditive.

The last property provides a mathematical diversification principle.


In the VaR counterexample,

$$
\operatorname{CVaR}_{0.95}(X)
=
\operatorname{CVaR}_{0.95}(Y)
=
0.8,
$$

while

$$
\boxed{
\operatorname{CVaR}_{0.95}(X+Y)
=
1.032
\le1.6.
}
$$


## 11. Optimal excess-of-loss retention

For an exponential severity $X\sim\operatorname{Exp}(\beta)$, the chapter studies

$$
\boxed{
J(d)
=
\operatorname{CVaR}_\alpha(X\wedge d)
+
(1+\theta)E[(X-d)^+].
}
$$

The first term measures retained tail risk; the second is the reinsurance premium under an expected-value loading.


Let

$$
q_\alpha
=
-\frac{\log(1-\alpha)}{\beta}.
$$

For $d\le q_\alpha$, the capped retained loss satisfies

$$
\operatorname{CVaR}_\alpha(X\wedge d)=d,
$$

and therefore

$$
J(d)
=
d
+
(1+\theta)\frac{e^{-\beta d}}{\beta}.
$$

The stationary point is

$$
\boxed{
d_0
=
\frac{\log(1+\theta)}{\beta}.
}
$$


The three regimes are determined by

$$
(1+\theta)(1-\alpha).
$$

If

$$
\boxed{
(1+\theta)(1-\alpha)<1,
}
$$

then the finite optimizer is

$$
\boxed{
d^*
=
\frac{\log(1+\theta)}{\beta}.
}
$$

If equality holds, the objective is flat beyond $q_\alpha$.

If

$$
(1+\theta)(1-\alpha)>1,
$$

the infimum corresponds to retaining the entire loss, i.e. no reinsurance.


In [ ]:
opt_mean = widgets.FloatSlider(value=2000,min=500,max=10000,step=500,description="mean")
opt_alpha = widgets.FloatSlider(value=0.95,min=0.8,max=0.995,step=0.005,description="alpha")
opt_theta = widgets.FloatSlider(value=0.25,min=0,max=5,step=0.05,description="theta")
opt_output = widgets.Output()

def update_retention(*_):
    with opt_output:
        clear_output(wait=True)

        beta = 1/opt_mean.value
        alpha = opt_alpha.value
        theta = opt_theta.value

        regime, dstar = exp_retention_regime(beta,alpha,theta)
        q = exponential_var(alpha,beta)

        display(Markdown(f"Regime: **{regime}**"))
        display(Math(r"q_\alpha=" + f"{q:.4f}"))

        if math.isfinite(dstar):
            display(Math(r"d^*=" + f"{dstar:.4f}"))

        upper = max(2*q, 5*opt_mean.value)
        dgrid = np.linspace(0,upper,700)
        J = np.array([
            retention_objective(d,alpha,beta,theta)
            for d in dgrid
        ])

        fig, ax = plt.subplots(figsize=(8,3.4))
        ax.plot(dgrid,J)
        ax.axvline(q,linestyle="--",label="VaR cutoff")
        if math.isfinite(dstar):
            ax.axvline(dstar,linestyle=":",label="selected retention")
        ax.set_xlabel("retention d")
        ax.set_ylabel("objective J(d)")
        ax.legend()
        ax.set_title("CVaR-plus-reinsurance-premium objective")
        plt.show()

for c in (opt_mean,opt_alpha,opt_theta):
    c.observe(update_retention,names="value")

display(widgets.VBox([
    widgets.HBox([opt_mean,opt_alpha,opt_theta]),
    opt_output,
]))
update_retention()


### Chapter benchmark

For mean severity $2000$, $\alpha=0.95$ and loading $\theta=0.25$,

$$
(1+\theta)(1-\alpha)
=
0.0625<1,
$$

so

$$
\boxed{
d^*
=
2000\log(1.25)
\approx446.29.
}
$$

This is a **per-loss** retention under the stated objective, not a universal optimal contract and not an annual aggregate attachment point.


### Portfolio-level consequence

If the selected per-loss retention is $d^*$,

$$
R=X\wedge d^*,
\qquad
C=(X-d^*)^+.
$$

For exponential severity,

$$
E[R]
=
\frac{1-e^{-\beta d^*}}{\beta},
$$

$$
E[C]
=
\frac{e^{-\beta d^*}}{\beta},
$$

and

$$
E[R^2]
=
\frac{2}{\beta^2}
\left[
1-e^{-\beta d^*}(1+\beta d^*)
\right].
$$

For annual Poisson frequency $\lambda$,

$$
E[S_R]
=
\lambda E[R],
$$

and

$$
\operatorname{Var}(S_R)
=
\lambda E[R^2].
$$


## 12. Historical problem: the Cramér--Lundberg risk process

Let claims arrive according to a Poisson process of rate $\lambda$, with iid severities $X_i$.

Define aggregate claims

$$
S(t)
=
\sum_{i=1}^{N(t)}X_i,
$$

and surplus

$$
\boxed{
U(t)
=
u+ct-S(t).
}
$$

The ruin time is

$$
\boxed{
\tau_u
=
\inf\{
t\ge0:
U(t)<0
\},
}
$$

and the ultimate ruin probability is

$$
\boxed{
\psi(u)
=
P(\tau_u<\infty).
}
$$


If

$$
\mu=E[X_1],
$$

then

$$
E[S(t)]
=
\lambda t\mu,
$$

and

$$
\boxed{
E[U(t)]
=
u+
(c-\lambda\mu)t.
}
$$

The expected surplus has positive drift precisely when

$$
\boxed{
c>\lambda\mu.
}
$$


### Safety loading

In the nondegenerate case, write

$$
\boxed{
c
=
(1+\eta)\lambda\mu,
\qquad
\eta>0.
}
$$

The parameter $\eta$ is the safety loading.


### Strong long-run drift

If $\lambda>0$ and $E[X_1]=\mu<\infty$,

$$
\boxed{
\frac{S(t)}{t}
\xrightarrow{\mathrm{a.s.}}
\lambda\mu,
}
$$

and

$$
\boxed{
\frac{U(t)}{t}
\xrightarrow{\mathrm{a.s.}}
c-\lambda\mu.
}
$$

If

$$
c<\lambda\mu,
$$

then $U(t)\to-\infty$ almost surely and ultimate ruin occurs with probability one.


A positive safety loading does **not** imply zero ruin probability.

Even when

$$
c>\lambda\mu,
$$

a sufficiently large early claim can push the surplus below zero.


In [ ]:
risk_T = widgets.FloatSlider(value=10,min=2,max=30,step=1,description="years")
risk_c = widgets.FloatSlider(value=28000,min=15000,max=40000,step=1000,description="premium rate")
risk_output = widgets.Output()

def update_risk_path(*_):
    with risk_output:
        clear_output(wait=True)

        T = risk_T.value
        c = risk_c.value
        u0 = 10000.0
        lam = 12.0
        mean_sev = 2000.0

        rng = np.random.default_rng(2026)

        # Simulate claim times through exponential interarrivals.
        t = 0.0
        times = []
        amounts = []

        while True:
            t += rng.exponential(1/lam)
            if t > T:
                break
            times.append(t)
            amounts.append(rng.exponential(mean_sev))

        grid = np.linspace(0,T,1200)
        cumulative = np.zeros_like(grid)

        for ti, xi in zip(times,amounts):
            cumulative += xi*(grid >= ti)

        surplus = u0 + c*grid - cumulative

        fig, ax = plt.subplots(figsize=(8,3.5))
        ax.plot(grid,surplus)
        ax.axhline(0,linestyle="--")
        ax.set_xlabel("time")
        ax.set_ylabel("surplus")
        ax.set_title("One Cramer-Lundberg surplus path")
        plt.show()

        display(Math(r"c-\lambda\mu=" + f"{c-lam*mean_sev:,.2f}"))

        ruin_idx = np.where(surplus < 0)[0]
        if ruin_idx.size:
            display(Markdown(f"Ruin occurred on this simulated path by approximately **t={grid[ruin_idx[0]]:.3f}**."))
        else:
            display(Markdown("No ruin occurred on this finite simulated horizon."))

for c in (risk_T,risk_c):
    c.observe(update_risk_path,names="value")

display(widgets.VBox([
    widgets.HBox([risk_T,risk_c]),
    risk_output,
]))
update_risk_path()


## 13. Complete portfolio example

Suppose annual claim count is

$$
N\sim\operatorname{Poisson}(12),
$$

and severity is exponential with mean $2000$.

Then

$$
\boxed{
E[S]=24{,}000,
}
$$

and

$$
\boxed{
\operatorname{Var}(S)=96{,}000{,}000.
}
$$

With a $20\%$ expected-value premium loading,

$$
\boxed{
\Pi_{\mathrm{EV}}(S)=28{,}800.
}
$$

At $\alpha=0.95$,

$$
\operatorname{VaR}_{0.95}(X)
\approx5991.46,
$$

and

$$
\operatorname{CVaR}_{0.95}(X)
\approx7991.46.
$$


In [ ]:
mean_severity = 2000.0
beta = 1/mean_severity
alpha = 0.95
theta = 0.25
lam = 12

dstar = math.log(1+theta)/beta
mean_R = (1-math.exp(-beta*dstar))/beta
mean_C = math.exp(-beta*dstar)/beta
second_R = (
    2/beta**2
    *
    (
        1
        -
        math.exp(-beta*dstar)*(1+beta*dstar)
    )
)

display(Math(r"d^*=" + f"{dstar:.4f}"))
display(Math(r"E[S_R]=" + f"{lam*mean_R:,.2f}"))
display(Math(r"\operatorname{Var}(S_R)=" + f"{lam*second_R:,.2f}"))
display(Math(r"\text{annual ceded expectation}=" + f"{lam*mean_C:,.2f}"))
display(Math(r"\text{annual reinsurance premium}=" + f"{1.25*lam*mean_C:,.2f}"))


## 14. Python laboratory: aggregate loss and reinsurance


In [ ]:
lab_N = widgets.IntSlider(value=50000,min=5000,max=150000,step=5000,description="years")
lab_output = widgets.Output()

def update_actuarial_lab(*_):
    with lab_output:
        clear_output(wait=True)

        n_years = lab_N.value
        lam = 12.0
        mean_severity = 2000.0
        beta = 1/mean_severity
        alpha = 0.95
        theta = 0.25

        N, S = simulate_compound_poisson(
            lam,
            mean_severity,
            n_years,
            seed=12345,
        )

        theory_mean = lam*mean_severity
        theory_var = lam*2*mean_severity**2

        var_hat = empirical_var(S,alpha)
        cvar_hat = empirical_cvar(S,alpha)

        dgrid = np.linspace(0,12000,300)
        J = np.array([
            retention_objective(d,alpha,beta,theta)
            for d in dgrid
        ])
        d_grid = dgrid[np.argmin(J)]
        d_exact = math.log(1+theta)/beta

        rows = [
            "| diagnostic | empirical / numerical | theory |",
            "|---|---:|---:|",
            f"| aggregate mean | {S.mean():.2f} | {theory_mean:.2f} |",
            f"| aggregate variance | {S.var():.2f} | {theory_var:.2f} |",
            f"| aggregate VaR 0.95 | {var_hat:.2f} | simulation target |",
            f"| aggregate CVaR 0.95 | {cvar_hat:.2f} | simulation target |",
            f"| per-loss grid optimizer | {d_grid:.2f} | {d_exact:.2f} |",
        ]

        display(Markdown("\n".join(rows)))

lab_N.observe(update_actuarial_lab,names="value")
display(widgets.VBox([lab_N,lab_output]))
update_actuarial_lab()


## 15. Guided exercise generator


In [ ]:
exercise_rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random","random"),
        ("Compound Poisson","cp"),
        ("Deductible","ded"),
        ("Quota share","quota"),
        ("VaR","var"),
        ("CVaR","cvar"),
        ("Retention","ret"),
        ("Risk process","risk"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_output = widgets.Output()
feedback_output = widgets.Output()
state = {}

def make_exercise(_=None):
    kind = exercise_kind.value
    if kind == "random":
        kind = exercise_rng.choice(["cp","ded","quota","var","cvar","ret","risk"])

    if kind == "cp":
        target = "20000000"
        prompt = "N~Poisson(10), severity mean 1000 with second moment 2,000,000. Find Var(S)."
        hint = "Compound Poisson variance is lambda E[X^2]."
        solution = r"\operatorname{Var}(S)=20{,}000{,}000."

    elif kind == "ded":
        target = str(2000*math.exp(-0.5))
        prompt = "X~Exp(mean 2000). Find E[(X-1000)^+] as a decimal."
        hint = "Use mean times exp(-d/mean)."
        solution = r"2000e^{-1/2}."

    elif kind == "quota":
        target = "10562500"
        prompt = "Var(X)=25,000,000 and q=0.35 is ceded. Find retained variance."
        hint = "Retained proportion is 0.65 and variance scales quadratically."
        solution = r"0.65^2(25{,}000{,}000)=10{,}562{,}500."

    elif kind == "var":
        target = str(1000*math.log(100))
        prompt = "X~Exp(mean 1000). Find VaR_0.99 as a decimal."
        hint = "Use -mean log(1-alpha)."
        solution = r"1000\log(100)."

    elif kind == "cvar":
        target = str(1000*math.log(100)+1000)
        prompt = "Continue the previous exponential loss. Find CVaR_0.99 as a decimal."
        hint = "Exponential CVaR equals VaR plus one mean."
        solution = r"1000\log(100)+1000."

    elif kind == "ret":
        target = str(2000*math.log(1.25))
        prompt = "Mean severity 2000, alpha=0.95, theta=0.25. Find the finite optimal retention."
        hint = "The finite regime holds and d*=mean*log(1+theta)."
        solution = r"d^*=2000\log(1.25)."

    else:
        target = "yes"
        prompt = "If c<lambda*mu in the Cramer-Lundberg model, does ultimate ruin occur with probability one under the chapter assumptions? yes/no"
        hint = "Use the strong long-run drift theorem."
        solution = r"\text{Yes.}"

    state.clear()
    state.update(target=target,hint=hint,solution=solution)
    answer_box.value = ""

    with prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n"+prompt))

    with feedback_output:
        clear_output(wait=True)

def show_hint(_):
    with feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** "+state["hint"]))

def reveal(_):
    with feedback_output:
        clear_output(wait=True)
        display(Math(state["solution"]))

def check(_):
    with feedback_output:
        clear_output(wait=True)
        guess = answer_box.value.strip().lower().replace(" ","")
        target = state["target"].strip().lower().replace(" ","")
        correct = guess == target
        if not correct:
            try:
                correct = abs(float(guess)-float(target)) < 1e-3
            except Exception:
                pass
        display(Markdown("**Correct.**" if correct else "**Not yet. Check the model level, transform and integrability assumptions.**"))

new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([exercise_kind,new_button]),
    prompt_output,
    widgets.HBox([answer_box,check_button]),
    widgets.HBox([hint_button,reveal_button]),
    feedback_output,
]))

make_exercise()


## 16. AI Audit: actuarial claims

Audit each claim before accepting it.

1. “For a compound Poisson aggregate loss, $\operatorname{Var}(S)=\lambda\operatorname{Var}(X)$.”
2. “A $99\%$ VaR is a hard upper bound that the loss cannot exceed.”
3. “For every distribution, including distributions with atoms, $\operatorname{CVaR}_\alpha(X)=E[X\mid X>\operatorname{VaR}_\alpha(X)]$.”
4. “A positive safety loading $c>\lambda\mu$ makes ultimate ruin impossible.”
5. “Per-loss excess-of-loss and aggregate stop-loss are the same contract.”
6. “The net premium must exceed expected loss.”
7. “Under quota share, retaining $50\%$ of each loss retains $50\%$ of the variance.”
8. “VaR is always subadditive.”
9. “The characteristic-function random-sum formula requires an MGF.”
10. “The optimal retention $d^*$ is a universal actuarial recommendation.”

The correct repairs are:

- compound Poisson variance is $\lambda E[X^2]$;
- VaR is a quantile cutoff, not a maximum possible loss;
- the conditional-tail interpretation of CVaR needs continuity/regularity, while the optimization definition works generally;
- positive safety loading means positive drift, not zero ruin probability;
- per-loss and aggregate treaties apply the same algebra to different random variables;
- net premium equals $E[X]$;
- variance scales with the square of the retained fraction;
- VaR can fail subadditivity;
- characteristic functions always exist;
- retention optimization is relative to its chosen severity model, risk measure and premium principle.


### Suggested AI-guided activities

- Ask an AI system to derive the random-sum mean and variance from conditional expectation and conditional variance, then check every independence step.
- Ask it to compare the five premium principles and state the integrability requirement of each.
- Ask it to construct the VaR subadditivity counterexample and then compute CVaR for the same pair.
- Ask it to solve the three retention regimes for the exponential model without skipping the case split at $q_\alpha$.
- Ask it to explain why $c>\lambda\mu$ is a drift statement rather than a pathwise guarantee.


## 17. Self-check quiz


In [ ]:
quiz_data = [
    ("1. Net premium equals:", ["Choose...","E[X]","VaR","Var(X)"], "E[X]", r"\Pi_0(X)=E[X]."),
    ("2. Compound Poisson variance equals:", ["Choose...","lambda E[X^2]","lambda Var(X)"], "lambda E[X^2]", r"\operatorname{Var}(S)=\lambda E[X^2]."),
    ("3. Stop-loss transform is:", ["Choose...","E[(X-d)^+]","E[X wedge d]"], "E[(X-d)^+]", r"\pi_X(d)=E[(X-d)^+]."),
    ("4. Quota-share retained variance scales:", ["Choose...","linearly","quadratically"], "quadratically", r"\operatorname{Var}((1-q)X)=(1-q)^2\operatorname{Var}(X)."),
    ("5. VaR is always subadditive:", ["Choose...","true","false"], "false", r"\text{The chapter gives a discrete counterexample.}"),
    ("6. General CVaR is defined through:", ["Choose...","positive-part optimization","conditional mean beyond VaR only"], "positive-part optimization", r"\operatorname{CVaR}_\alpha=\inf_z\{z+(1-\alpha)^{-1}E[(X-z)^+]\}."),
    ("7. CVaR is coherent:", ["Choose...","true","false"], "true", r"\text{It is monotone, translation invariant, homogeneous and subadditive.}"),
    ("8. Positive safety loading makes ruin impossible:", ["Choose...","true","false"], "false", r"\text{Large early claims may still cause ruin.}"),
    ("9. If c<lambda mu, U(t)/t tends almost surely to:", ["Choose...","a negative number","zero","a positive number"], "a negative number", r"U(t)/t\to c-\lambda\mu<0."),
    ("10. A per-loss retention optimizer is automatically an annual aggregate attachment point:", ["Choose...","true","false"], "false", r"\text{The levels of aggregation differ.}"),
]

quiz_widgets = []
rows = []

for prompt, options, _, _ in quiz_data:
    d = widgets.Dropdown(options=options,value="Choose...",layout=widgets.Layout(width="480px"))
    quiz_widgets.append(d)
    rows.append(widgets.HBox([widgets.HTML(f"<div style='width:720px'>{prompt}</div>"),d]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()

def grade(_):
    with quiz_output:
        clear_output(wait=True)
        score = sum(w.value == correct for w,(_,_,correct,_) in zip(quiz_widgets,quiz_data))
        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))
        for i,(w,(_,_,correct,explanation)) in enumerate(zip(quiz_widgets,quiz_data),1):
            mark = "✓" if w.value == correct else "✗"
            display(Markdown(f"**{mark} Question {i}:** correct answer = `{correct}`"))
            display(Math(explanation))

grade_button.on_click(grade)
display(widgets.VBox(rows+[grade_button,quiz_output]))


## 18. Automatic mathematical verification


In [ ]:
# Exponential tail formulas.
beta = 1/2000
d = 1000
assert abs(
    stop_loss_exponential(d,beta)
    -
    2000*math.exp(-0.5)
) < 1e-10

assert abs(
    limited_expected_exponential(d,beta)
    +
    stop_loss_exponential(d,beta)
    -
    2000
) < 1e-10

# Deductible + limit formula.
u = 500
expected = capped_excess_expected_exponential(d,u,beta)
manual = (
    math.exp(-beta*d)
    -
    math.exp(-beta*(d+u))
)/beta
assert abs(expected-manual) < 1e-12

# Compound Poisson benchmark.
mean_s, var_s = compound_poisson_moments(
    10,
    1000,
    2_000_000,
)
assert abs(mean_s-10_000) < 1e-12
assert abs(var_s-20_000_000) < 1e-12

# Quota share.
assert abs(0.65*10_000-6500) < 1e-12
assert abs(0.65**2*25_000_000-10_562_500) < 1e-8

# Exponential VaR/CVaR.
q = exponential_var(0.99,1/1000)
c = exponential_cvar(0.99,1/1000)
assert abs(q-1000*math.log(100)) < 1e-10
assert abs(c-q-1000) < 1e-10

# VaR counterexample probabilities.
assert 0.96**2 < 0.95
assert 1-0.04**2 > 0.95

# CVaR counterexample.
assert abs(0.04/0.05-0.8) < 1e-12
assert abs(1+0.0016/0.05-1.032) < 1e-12
assert 1.032 <= 1.6

# Retention benchmark.
beta = 1/2000
alpha = 0.95
theta = 0.25
regime, dstar = exp_retention_regime(beta,alpha,theta)
assert regime == "finite"
assert abs(dstar-2000*math.log(1.25)) < 1e-10

# Classical risk process negative drift example.
lam = 12
mu = 2000
premium_rate = 22_000
assert premium_rate-lam*mu == -2000

show_result(
    "All Chapter 18 automatic checks passed",
    r"E[S]=E[N]E[X]",
    r"\operatorname{Var}(S_{\mathrm{CP}})=\lambda E[X^2]",
    r"E[(X-d)^+]=\int_d^\infty\overline F_X(x)\,dx",
    r"\operatorname{CVaR}_\alpha(X)=\inf_z\left\{z+\frac{E[(X-z)^+]}{1-\alpha}\right\}",
    r"\frac{U(t)}t\to c-\lambda\mu\quad\text{a.s.}",
)


## 19. Chapter map

| Concept | Computational representation |
|---|---|
| individual loss | survival curve |
| tail integral | area under survival function |
| premium principles | interactive exponential premium |
| random sum | total-expectation/variance calculator |
| random-sum transforms | PGF/MGF/CF composition |
| compound Poisson | simulation and moment check |
| compound Poisson process | time-scaled moments and martingale |
| deductible / limit | positive-part and truncation formulas |
| stop-loss transform | decreasing convex curve |
| quota share | retained mean and variance |
| excess of loss | retained/ceded decomposition |
| VaR | exponential quantile |
| VaR non-subadditivity | discrete diversification counterexample |
| CVaR | positive-part optimization |
| quantile minimizers | one-sided cdf conditions |
| average quantiles | integral of the quantile function |
| CVaR coherence | diversification-friendly structure |
| retention optimization | three exponential regimes |
| portfolio consequence | retained compound Poisson moments |
| Cramér--Lundberg | simulated surplus path |
| long-run drift | strong-law interpretation |
| AI Audit | actuarial model and risk-measure checks |

The final chapter closes the book by reusing almost every major probability theme:

$$
\boxed{
\text{expectation, transforms, conditional expectation, Poisson processes,}
}
$$

$$
\boxed{
\text{quantiles, tail integrals, martingales and limit theorems.}
}
$$
